# Rag_pipeline_**Member1**

In [ ]:
# ── Upload WikiBio dataset to Colab runtime ───────────────────
#
from google.colab import files
import shutil, os

if not os.path.exists("wikibio_gpt4o.json"):
    print("📤 Please upload wikibio_gpt4o.json ...")
    uploaded = files.upload()
    print("✅ Uploaded:", list(uploaded.keys()))
else:
    print("✅ wikibio_gpt4o.json already present")

embedding generation and storing it to faiss DB

In [ ]:
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq
from datasets import load_dataset

groq_client = Groq(api_key=GROQ_API_KEY)

# ── Global Setup ──────────────────────────────────────────────
print("🔢 Loading embedding model (multi-qa-MiniLM-L6-cos-v1)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    model_kwargs={"device": "cpu"}
)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
BATCH = 500

# ══════════════════════════════════════════════════════════════
#  DATABASE 1: SQuAD (General Knowledge)
# ══════════════════════════════════════════════════════════════
print("\n📥 Loading SQuAD dataset from HuggingFace...")
squad = load_dataset("squad", split="validation")

# We extract 200 unique contexts to keep the index fast and lightweight for the demo
unique_squad_contexts = list(set(squad["context"]))[:200]
squad_docs = []

for ctx in unique_squad_contexts:
    for chunk in splitter.split_text(ctx):
        squad_docs.append(Document(page_content=chunk, metadata={"source": "squad"}))

print(f"📄 {len(squad_docs)} SQuAD chunks created.")
print("🏗️ Building SQuAD FAISS index...")

squad_vectorstore = None
for i in range(0, len(squad_docs), BATCH):
    batch = squad_docs[i:i+BATCH]
    if squad_vectorstore is None:
        squad_vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        squad_vectorstore.merge_from(FAISS.from_documents(batch, embeddings))
    print(f"  Processed {min(i+BATCH, len(squad_docs))}/{len(squad_docs)} SQuAD chunks")

squad_vectorstore.save_local("squad_faiss_index")
print("✅ SQuAD FAISS index saved to 'squad_faiss_index'")


# ══════════════════════════════════════════════════════════════
#  DATABASE 2: WikiBio (Biographies)
# ══════════════════════════════════════════════════════════════
print("\n📥 Loading WikiBio GPT-4o dataset...")
try:
    with open("wikibio_gpt4o.json", "r") as f:
        wikibio_data = json.load(f)
    print(f"✅ Loaded {len(wikibio_data)} WikiBio items")

    wikibio_docs = []
    for item in wikibio_data:
        # Use the first 5 reference samples per item as retrieval corpus
        for sample in item["text_samples"][:5]:
            for chunk in splitter.split_text(sample):
                wikibio_docs.append(Document(
                    page_content=chunk,
                    metadata={"unique_id": item.get("unique_id", ""), "source": "wikibio"}
                ))

    print(f"📄 {len(wikibio_docs)} WikiBio chunks created.")
    print("🏗️ Building WikiBio FAISS index...")

    wikibio_vectorstore = None
    for i in range(0, len(wikibio_docs), BATCH):
        batch = wikibio_docs[i:i+BATCH]
        if wikibio_vectorstore is None:
            wikibio_vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            wikibio_vectorstore.merge_from(FAISS.from_documents(batch, embeddings))
        print(f"  Processed {min(i+BATCH, len(wikibio_docs))}/{len(wikibio_docs)} WikiBio chunks")

    wikibio_vectorstore.save_local("wikibio_faiss_index")
    print("✅ WikiBio FAISS index saved to 'wikibio_faiss_index'")

except FileNotFoundError:
    print("⚠️ ERROR: 'wikibio_gpt4o.json' not found. Please upload it to your Colab files to build the WikiBio index.")


def retriever_agent(question: str, k: int = 5) -> str:
    results = vectorstore.similarity_search(question, k=k)
    return "\n\n".join(f"[Passage {i+1}]\n{d.page_content}" for i, d in enumerate(results))


def generator_agent(question: str, context: str) -> str:
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": """You are a helpful answering agent.
Answer the question completely and concisely.
Use the provided context as your primary source."""},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"}
        ],
        temperature=0.7,
        max_tokens=200
    )
    return resp.choices[0].message.content.strip()


print("✅ Member 1 module ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MEMBER 2 MODULE — Hallucination Detection (HDM)
# ══════════════════════════════════════════════════════════════
import json
import re
import time
import numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("📦 Loading sentence-transformer (multi-qa-MiniLM-L6-cos-v1)...")
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")
print("✅ Embedding model loaded")


print("📦 Loading NLI model...")
nli_model = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=-1  # CPU
)
print("✅ NLI model loaded")


def nli_score(context: str, claim: str) -> float:
    """
    Returns a hallucination signal: 1.0 if context CONTRADICTS claim,
    0.5 if NEUTRAL, 0.0 if ENTAILS.
    Truncates to 512 tokens to avoid model limits.
    """
    try:
        # Use first 800 chars of context to stay within token limit
        result = nli_model(
            f"{context[:800]} [SEP] {claim}",
            truncation=True,
            max_length=512
        )[0]
        label = result["label"].upper()
        if label == "CONTRADICTION":
            return 1.0
        elif label == "NEUTRAL":
            return 0.5
        else:  # ENTAILMENT
            return 0.0
    except Exception:
        return 0.5  # neutral fallback

TRIPLET_PROMPT = """Extract factual claims from the text below as Subject-Predicate-Object triplets.
You MUST return ONLY a raw JSON array. No keys like "results", no markdown, no explanation.
Start your response with [ and end with ].

Example output:
[{{"subject": "einstein", "predicate": "developed", "object": "general relativity"}}]

TEXT TO PROCESS:
{text}"""


def extract_triplets(client, text, retries=2):
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": "You are a JSON generator. You only output valid JSON arrays. Never output anything else."},
                    {"role": "user",   "content": TRIPLET_PROMPT.format(text=text[:1500])}
                ],
                temperature=0.0,
                max_tokens=1024
            )
            raw = r.choices[0].message.content.strip()
            raw = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")

            # Try direct JSON parse first
            try:
                parsed = json.loads(raw)
                if isinstance(parsed, dict):
                    for key in ["results", "triplets", "data", "output"]:
                        if key in parsed and isinstance(parsed[key], list):
                            parsed = parsed[key]
                            break
                    else:
                        parsed = list(parsed.values())[0] if parsed else []
                if isinstance(parsed, list):
                    valid = [
                        {"subject":   str(t["subject"]).lower().strip(),
                         "predicate": str(t["predicate"]).lower().strip(),
                         "object":    str(t["object"]).lower().strip()}
                        for t in parsed
                        if isinstance(t, dict) and all(k in t for k in ["subject", "predicate", "object"])
                    ]
                    if valid:
                        return valid
            except json.JSONDecodeError:
                pass

            # Fallback: regex to find array
            m = re.search(r"\[.*\]", raw, re.DOTALL)
            if not m:
                print(f"Warning: No JSON array found (attempt {attempt+1})")
                continue
            parsed = json.loads(m.group(0))
            return [
                {"subject":   str(t["subject"]).lower().strip(),
                 "predicate": str(t["predicate"]).lower().strip(),
                 "object":    str(t["object"]).lower().strip()}
                for t in parsed
                if isinstance(t, dict) and all(k in t for k in ["subject", "predicate", "object"])
            ]

        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 30
                print(f"⏳ Rate limit hit — waiting {wait}s...")
                time.sleep(wait)
                continue
            print(f"Warning: Triplet extraction failed. Error: {e}")
            return []
    return []


def build_kg(triplets: list, name="KG") -> nx.DiGraph:
    G = nx.DiGraph(name=name)
    for t in triplets:
        s, p, o = t["subject"].lower(), t["predicate"].lower(), t["object"].lower()
        G.add_node(s)
        G.add_node(o)
        G.add_edge(s, o, predicate=p)
    return G


def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

def hallucination_detector(client, context: str, answer: str,
                           threshold: float = 0.30) -> dict:
    """
    Hybrid scorer: KG cosine similarity + NLI contradiction signal.
    score = 0.6 * (1 - cosine_sim) + 0.4 * nli_score
    """
    ctx_triplets = extract_triplets(client, context[:1500])
    ans_triplets = extract_triplets(client, answer[:800])

    print(f"  Triplets — context: {len(ctx_triplets)}, answer: {len(ans_triplets)}")

    G_ctx = build_kg(ctx_triplets, "Context_KG")
    G_ans = build_kg(ans_triplets, "Answer_KG")

    # Save context edges list so we can access the (cu, cv) entities for the penalty
    ctx_edges_list = list(G_ctx.edges(data=True))
    ctx_texts = [f"{u} {d['predicate']} {v}" for u, v, d in ctx_edges_list]
    ctx_embs  = embedding_model.encode(ctx_texts) if ctx_texts else []

    supported, unsupported = [], []

    for u, v, data in G_ans.edges(data=True):
        edge_text = f"{u} {data['predicate']} {v}"

        # ── Signal 1: KG cosine similarity WITH STRICT ENTITY PENALTY ──
        best_cos, best_match = 0.0, ""
        if ctx_texts:
            emb = embedding_model.encode(edge_text)
            for ce, ct, (cu, cv, cdata) in zip(ctx_embs, ctx_texts, ctx_edges_list):
                s = cosine_sim(emb, ce)

                # --- EXACT ENTITY MATCH PENALTY (STRICT AND LOGIC) ---
                u_safe, v_safe = str(u).lower(), str(v).lower()
                cu_safe, cv_safe = str(cu).lower(), str(cv).lower()

                # Did it find a match for the Subject? (Checking both context subject and object)
                subj_match = (u_safe in cu_safe or cu_safe in u_safe or u_safe in cv_safe or cv_safe in u_safe)

                # Did it find a match for the Object? (Checking both context subject and object)
                obj_match = (v_safe in cv_safe or cv_safe in v_safe or v_safe in cu_safe or cu_safe in v_safe)

                # BOTH the subject and the object MUST match. If either is hallucinated, slash the score!
                if not (subj_match and obj_match):
                    s = s * 0.2  # Slashed by 80% to absolutely guarantee it fails the threshold
                # -----------------------------------------------------

                if s > best_cos:
                    best_cos, best_match = s, ct

        # ── Signal 2: NLI on the SPECIFIC CLAIM ────────
        # Evaluates just the specific triplet claim against the context.
        nli = nli_score(context, edge_text)

        # ── Hybrid score: high = likely hallucinated ─────────
        hybrid = 0.6 * (1.0 - best_cos) + 0.4 * nli

        rec = {
            "subject": u, "predicate": data["predicate"], "object": v,
            "claim_text": edge_text,
            "similarity_score": round(best_cos, 4),
            "nli_score": round(nli, 4),
            "hybrid_score": round(hybrid, 4),
            "best_context_match": best_match
        }
        (supported if hybrid < threshold else unsupported).append(rec)

    total = len(supported) + len(unsupported)
    score = round(len(unsupported) / total, 4) if total else 0.0

    return {
        "hallucination_score": score,
        "unsupported_claims":  unsupported,
        "supported_claims":    supported,
        "context_triplets":    ctx_triplets,
        "answer_triplets":     ans_triplets,
        "context_graph":       G_ctx,
        "answer_graph":        G_ans,
        "total_claims":        total
    }

print("✅ Member 2 module ready")

 **Member 3: Correction Agent + LangGraph Orchestrator**

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MEMBER 3 MODULE — Correction Agent + LangGraph
# ══════════════════════════════════════════════════════════════
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

CORRECTION_PROMPT = """You are a fact-correction agent. Your only job is to output a corrected answer.

RULES:
- Answer in ONE sentence only.
- Use ONLY facts present in the context below.
- Do NOT copy paste the context.
- Do NOT explain what you are doing.
- Do NOT say "the context says" or "based on the context".
- Just answer the question directly and concisely.

QUESTION: {question}
CONTEXT: {context}
ORIGINAL ANSWER (may contain errors): {answer}
FLAGGED UNSUPPORTED CLAIMS: {claims}

One sentence corrected answer:"""


def correction_agent(question: str, answer: str, context: str, unsupported: list) -> str:
    if not unsupported:
        return answer
    claims_text = "\n".join(f"  - {c['claim_text']}" for c in unsupported)
    resp = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": CORRECTION_PROMPT.format(
            question=question,
            answer=answer,
            context=context[:2000],
            claims=claims_text
        )}],
        temperature=0.0,
        max_tokens=256
    )
    return resp.choices[0].message.content.strip()


# ── LangGraph State ──────────────────────────────────────────
class RAGState(TypedDict):
    user_question:       str
    retrieved_context:   Optional[str]
    generated_answer:    Optional[str]
    hallucination_score: Optional[float]
    unsupported_claims:  Optional[list]
    supported_claims:    Optional[list]
    context_triplets:    Optional[list]
    answer_triplets:     Optional[list]
    corrected_answer:    Optional[str]
    final_answer:        Optional[str]
    verdict:             Optional[str]
    context_graph:       Optional[object]
    answer_graph:        Optional[object]


# ── LangGraph Nodes ──────────────────────────────────────────
def node_retrieve(state: RAGState) -> RAGState:
    print("  [Node: Retrieval Agent]")
    context = retriever_agent(state["user_question"], k=5)
    return {**state, "retrieved_context": context}


def node_generate(state: RAGState) -> RAGState:
    print("  [Node: Generator Agent]")
    answer = generator_agent(state["user_question"], state["retrieved_context"])
    return {**state, "generated_answer": answer}


def node_detect(state: RAGState) -> RAGState:
    print("  [Node: HDM Agent]")
    result = hallucination_detector(
        groq_client,
        state["retrieved_context"],
        state["generated_answer"]
    )
    return {
        **state,
        "hallucination_score": result["hallucination_score"],
        "unsupported_claims":  result["unsupported_claims"],
        "supported_claims":    result["supported_claims"],
        "context_triplets":    result["context_triplets"],
        "answer_triplets":     result["answer_triplets"],
        "context_graph":       result["context_graph"],
        "answer_graph":        result["answer_graph"]
    }


def node_verify(state: RAGState) -> RAGState:
    print(f"  [Node: Verification Agent] score={state['hallucination_score']}")
    return state


def node_correct(state: RAGState) -> RAGState:
    print("  [Node: Correction Agent]")
    corrected = correction_agent(
        state["user_question"],
        state["generated_answer"],
        state["retrieved_context"],
        state["unsupported_claims"]
    )
    return {**state, "corrected_answer": corrected,
            "final_answer": corrected, "verdict": "CORRECTED"}


def node_accept(state: RAGState) -> RAGState:
    print("  [Node: Accept]")
    return {**state, "final_answer": state["generated_answer"], "verdict": "ACCEPTED"}


def route_after_verify(state: RAGState) -> str:
    return "correct" if state["hallucination_score"] >= 0.3 else "accept"


# ── Build Graph ───────────────────────────────────────────────
workflow = StateGraph(RAGState)
workflow.add_node("retrieve", node_retrieve)
workflow.add_node("generate", node_generate)
workflow.add_node("detect",   node_detect)
workflow.add_node("verify",   node_verify)
workflow.add_node("correct",  node_correct)
workflow.add_node("accept",   node_accept)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "detect")
workflow.add_edge("detect",   "verify")
workflow.add_conditional_edges(
    "verify",
    route_after_verify,
    {"correct": "correct", "accept": "accept"}
)
workflow.add_edge("correct", END)
workflow.add_edge("accept",  END)
app = workflow.compile()
print("✅ LangGraph orchestrator ready")


# ── run_pipeline helper ───────────────────────────────────────
def run_pipeline(question: str) -> dict:
    print(f"\n{'='*60}")
    print(f"🚀 Running pipeline for: '{question}'")
    print('='*60)
    initial_state = {
        "user_question": question,
        "retrieved_context": None, "generated_answer": None,
        "hallucination_score": None, "unsupported_claims": None,
        "supported_claims": None, "context_triplets": None,
        "answer_triplets": None, "corrected_answer": None,
        "final_answer": None, "verdict": None,
        "context_graph": None, "answer_graph": None
    }
    result = app.invoke(initial_state)
    print(f"\n  Verdict : {result['verdict']}")
    print(f"  Score   : {result['hallucination_score']}")
    print(f"  Answer  : {result['final_answer']}")
    return result


print("✅ Member 3 module ready")

Quick Pipeline Test

In [ ]:
# Test with one clean and one hallucination-prone question
test_result = run_pipeline("In what country is Normandy located?")

In [ ]:
test_result2 = run_pipeline("Describe the Norman military tactics and their conquests")

In [ ]:
test_result3 = run_pipeline("Who built eiffel tower?")

 Evaluation (Paper Metrics)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  STEP 6 — EVALUATION using WikiBio (OPTIMIZED + STRICT SCORING)
# ══════════════════════════════════════════════════════════════
import json
import time
import os
from tqdm import tqdm

EVAL_ITEMS    = 50
HDM_THRESHOLD = 0.30
OUTPUT_FILE   = "wikibio_checkpoint.jsonl"

print(f"Evaluating {EVAL_ITEMS} WikiBio items. Results will auto-save to {OUTPUT_FILE}...\n")

# To prevent instantly hitting Groq's Token Limit, we pace the requests heavily.
# If you still hit limits, increase these numbers!
API_DELAY = 4.0

for item in tqdm(wikibio_data[:EVAL_ITEMS], desc="WikiBio items"):

    # 1. Build & Truncate Context to prevent Token overflow!
    # Taking only the first 1200 chars saves massive amounts of tokens.
    raw_context = "\n\n".join(item["text_samples"][:3])
    context = raw_context[:1200]

    sentences = item["original_text_sentences"]
    labels    = item["annotation"]
    unique_id = item["unique_id"]

    # 2. Extract Context Triplets ONCE per item
    ctx_triplets = []
    while True: # Infinite retry loop for rate limits
        try:
            ctx_triplets = extract_triplets(groq_client, context)
            break
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                print(f"\n⏳ Token Limit Hit on Context! Cooling down for 60 seconds...")
                time.sleep(60)
            else:
                print(f"\n⚠️ Context extraction error: {e}")
                break

    time.sleep(API_DELAY)

    # Pre-build Context Graph for scoring
    G_ctx      = build_kg(ctx_triplets, "Context_KG")
    ctx_edges  = list(G_ctx.edges(data=True))
    ctx_texts  = [f"{u} {d['predicate']} {v}" for u, v, d in ctx_edges]
    ctx_embs   = embedding_model.encode(ctx_texts) if ctx_texts else []

    # 3. Process each sentence
    for sent, label in zip(sentences, labels):
        sent = sent.strip()
        if not sent:
            continue

        ans_triplets = []
        while True:
            try:
                ans_triplets = extract_triplets(groq_client, sent)
                break
            except Exception as e:
                if "rate_limit" in str(e).lower() or "429" in str(e):
                    print(f"\n⏳ Token Limit Hit on Sentence! Cooling down for 60 seconds...")
                    time.sleep(60)
                else:
                    break

        # 4. Strict Scoring (Includes Exact Match Penalty + NLI)
        G_ans = build_kg(ans_triplets, "Answer_KG")
        supported, unsupported = [], []

        for u, v, data in G_ans.edges(data=True):
            edge_text  = f"{u} {data['predicate']} {v}"
            best_score = 0.0

            if ctx_texts:
                emb = embedding_model.encode(edge_text)
                for ce, ct, (cu, cv, cdata) in zip(ctx_embs, ctx_texts, ctx_edges):
                    s = cosine_sim(emb, ce)

                    # --- STRICT ENTITY PENALTY ---
                    u_safe, v_safe = str(u).lower(), str(v).lower()
                    cu_safe, cv_safe = str(cu).lower(), str(cv).lower()

                    subj_match = (u_safe in cu_safe or cu_safe in u_safe or u_safe in cv_safe or cv_safe in u_safe)
                    obj_match = (v_safe in cv_safe or cv_safe in v_safe or v_safe in cu_safe or cu_safe in v_safe)

                    if not (subj_match and obj_match):
                        s = s * 0.2 # Nuke the score if nouns don't match
                    # -----------------------------

                    if s > best_score:
                        best_score = s

            # --- NLI INTEGRATION ---
            nli_val = nli_score(context, edge_text)
            hybrid_score = 0.6 * (1.0 - best_score) + 0.4 * nli_val

            if hybrid_score < HDM_THRESHOLD:
                supported.append(edge_text)
            else:
                unsupported.append(edge_text)

        total = len(supported) + len(unsupported)
        final_score = round(len(unsupported) / total, 4) if total else 0.0
        predicted = "hallucinated" if final_score >= HDM_THRESHOLD else "accurate"

        record = {
            "unique_id": unique_id,
            "sentence": sent,
            "ground_truth": label,
            "predicted": predicted,
            "score": final_score
        }

        # 5. AUTO-SAVE IMMEDIATELY TO HARD DRIVE
        with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")

        time.sleep(API_DELAY)

print("\n✅ Evaluation completed and securely saved to your hard drive!")

Write app.py

In [ ]:
!pip install Groq
!pip install langchain-community

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

print("1. Caching raw SentenceTransformer...")
st_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

print("2. Caching LangChain HuggingFaceEmbeddings (This is what Streamlit was waiting for)...")
embed_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    model_kwargs={"device": "cpu"}
)

print("✅ All models perfectly cached! You are cleared to start Streamlit.")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  app.py [ANTI-DEADLOCK FIX VERSION]
# ══════════════════════════════════════════════════════════════

app_code = """
import os
# CRITICAL FIX: Stops HuggingFace from causing a multi-threading deadlock in Streamlit!
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import streamlit as st
import json, re, time
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from groq import Groq
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

st.set_page_config(page_title="RAG Hallucination Detector", page_icon="🔍", layout="wide")

st.markdown(\"\"\"
<style>
.score-chip { display:inline-block; padding:4px 14px; border-radius:20px;
              font-weight:600; font-size:13px; margin:4px 0; }
.score-good   { background:#d1fae5; color:#065f46; }
.score-medium { background:#fef3c7; color:#92400e; }
.score-bad    { background:#fee2e2; color:#991b1b; }
.triplet-row  { font-family:monospace; font-size:12px; padding:3px 8px;
                border-radius:4px; margin:2px 0; background:#f9fafb; }
.claim-supported   { color:#065f46; background:#d1fae5; }
.claim-unsupported { color:#991b1b; background:#fee2e2; font-weight:600; }
.db-squad   { background:#ede9fe; color:#4c1d95; display:inline-block;
              padding:3px 10px; border-radius:12px; font-size:12px; font-weight:500; }
.db-wikibio { background:#dbeafe; color:#1e3a5f; display:inline-block;
              padding:3px 10px; border-radius:12px; font-size:12px; font-weight:500; }
</style>
\"\"\", unsafe_allow_html=True)

# ── Session state ────────────────────────────────────────────
for k, v in [("chat_history", []), ("pipeline_results", [])]:
    if k not in st.session_state:
        st.session_state[k] = v

# ════════════════════════════════════════════════════════════
#  SAFE SINGLE-THREADED MODEL LOADING
# ════════════════════════════════════════════════════════════

@st.cache_resource(show_spinner="Loading Models & FAISS Indexes Safely...")
def load_all_resources():
    stores = {}

    # 1. Load Embeddings
    embed_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
        model_kwargs={"device": "cpu"}
    )

    # 2. Load Sentence Transformer
    st_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

    # 3. Load Databases
    for name, path in [
        ("SQuAD (General Knowledge)", "squad_faiss_index"),
        ("WikiBio (Biographies)",     "wikibio_faiss_index"),
    ]:
        try:
            stores[name] = FAISS.load_local(path, embed_model, allow_dangerous_deserialization=True)
        except Exception:
            stores[name] = None

    return embed_model, st_model, stores

# Load everything safely
embed_model, st_model, stores = load_all_resources()
groq_api_key = os.environ.get("GROQ_API_KEY", "")
groq_client  = Groq(api_key=groq_api_key) if groq_api_key else None

# ════════════════════════════════════════════════════════════
#  GROQ-BASED NLI (Zero Load Time)
# ════════════════════════════════════════════════════════════

NLI_PROMPT = \"\"\"You are a textual entailment classifier. Given a CONTEXT and a CLAIM, output exactly one word:
- entails      (the context confirms or supports the claim)
- neutral      (the context neither confirms nor contradicts the claim)
- contradicts  (the context directly contradicts the claim)

CONTEXT: {context}
CLAIM: {claim}

Output one word only (entails / neutral / contradicts):\"\"\"

def groq_nli(client, context: str, sentence: str) -> float:
    try:
        r = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "You are a textual entailment classifier. Output only one word."},
                {"role": "user", "content": NLI_PROMPT.format(context=context[:600], claim=sentence[:300])}
            ],
            temperature=0.0, max_tokens=5
        )
        label = r.choices[0].message.content.strip().lower()
        if "contradict" in label: return 1.0
        elif "entail" in label: return 0.0
        else: return 0.5
    except Exception:
        return 0.5

# ════════════════════════════════════════════════════════════
#  PIPELINE FUNCTIONS
# ════════════════════════════════════════════════════════════

def do_retrieve(vs, question, k=5):
    results = vs.similarity_search(question, k=k)
    return "\\n\\n".join(f"[Passage {i+1}]\\n{d.page_content}" for i, d in enumerate(results))

def do_generate(client, question, context):
    r = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are an answering agent. Answer completely and concisely using the provided context as your primary source."},
            {"role": "user", "content": f"Context:\\n{context}\\n\\nQuestion: {question}\\n\\nAnswer:"}
        ],
        temperature=0.3, max_tokens=200)
    return r.choices[0].message.content.strip()

TRIPLET_PROMPT = \"\"\"Extract factual claims from the text as Subject-Predicate-Object triplets.
Return ONLY a raw JSON array. No markdown, no explanation.
Example: [{{"subject": "einstein", "predicate": "developed", "object": "general relativity"}}]
TEXT: {text}\"\"\"

def extract_triplets(client, text, retries=2):
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[
                    {"role": "system", "content": "Output only a valid JSON array of SPO triplets."},
                    {"role": "user", "content": TRIPLET_PROMPT.format(text=text[:1200])}
                ],
                temperature=0.0, max_tokens=800)
            raw = re.sub(r"
http://googleusercontent.com/immersive_entry_chip/0

### Step 3: Run and Start Streamlit
After running the `app.py` block above, run your standard cell:
`!streamlit run app.py`

Once you open the ngrok tunnel, the app should finish loading almost instantly, and you will never see that infinite spinner again!

In [ ]:
from sentence_transformers import SentenceTransformer

print("Downloading the embedding model...")
# This will show a loading bar in Colab and take about 15 seconds
model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")
print("✅ Model successfully cached! You can now start Streamlit.")

Launch Streamlit App

In [ ]:
!pip install pyngrok
!pip install streamlit

In [ ]:
# FIXED VERSION

from pyngrok import ngrok
import subprocess, time, requests

NGROK_TOKEN = "3C88c0OmHFZmkMQdLfJw7hr0rD2_5FJKsbzXhfiacVH1pD9Zp"
ngrok.set_auth_token(NGROK_TOKEN)

# Kill old processes
ngrok.kill()
time.sleep(2)
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Start Streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8510",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for correct port
print("⏳ Waiting for Streamlit to start...")
for _ in range(15):
    time.sleep(2)
    try:
        r = requests.get("http://localhost:8510", timeout=2)  # ✅ FIXED
        if r.status_code == 200:
            print("✅ Streamlit is up!")
            break
    except:
        pass
else:
    print("⚠️ Streamlit may not have started.")
    print(proc.stderr.read().decode()[:500])

# Start ngrok
public_url = ngrok.connect(8510)
print(f"\n🌐 PUBLIC URL: {public_url}")